# ⚡ DeepSeek-V4 推理优化与部署

**本文目标**：理解 V4 推理栈的完整优化方案——从投机解码到 PD 分离。

读完这篇你会理解：
- 单层 MTP 投机解码的设计
- PD (Prefill-Decode) 分离部署架构
- CUDA Graph 内的元数据准备
- 分层多流重叠 (Hierarchical Multi-Stream Overlap)
- V4 在 B200/H200 上的实际性能数据

## 1. 投机解码: 单层 MTP 头

### 1.1 设计: 为什么只有一层？

```
标准 MTP (Multi-Token Prediction, V3 风格):
  多层的 draft model → 可以预测多个 token
  但每层都有独立的权重和计算 → overhead 大

V4 MTP:
  仅 1 层! → 一个单独训练的 DSv4 decoder layer
  仅运行 SWA 注意力 (无压缩器、无索引器)
  
  输入:
    h_proj: 前一步的 hidden state (投影到 MTP 维度)
    e_proj: 下一个 token 的 embedding (投影后)
    → 两者拼接/组合作为输入
  
  为什么只有一层?
    1. V4 本身已经非常深 → decoder 的单层计算已经能提供有用的 draft
    2. 简化设计 → 减少显存开销
    3. 仅 SWA (128 tokens) → 极低的计算量
    4. 投机解码的 accept rate 不需要太高, 1 层足够
  
  效果:
    Pro B200 (EAGLE-3): accept ~2.5 tokens/step  (vs V4 MTP 的 ~1.19)
    → EAGLE 更好, 但 MTP 更简单且不需要额外的 draft model 权重
```

### 1.2 投机解码中的 ShadowRadix 环扩展

```
投机解码的环 (ring) 问题:

  投机解码的 draft model 会猜测 N 个 future token
  → 这些 token 生成后进入 KV Cache
  → 如果 draft 被主模型 reject → 需要回滚
  
回滚可能覆盖 ShadowRadix 中的活跃数据!

解决方案: 环大小翻倍
  C4 ring:  8 → 16
  C128 ring: 128 → 256
  
  更大的环 = 更多的缓冲空间
  → 即使回滚, 也只是覆盖"刚写入但尚未被确认"的区域
  → 真正的活跃数据 (被 full_lock_ref 保护) 不受影响
```

## 2. PD 分离部署

### 2.1 为什么需要分离？

```
传统部署: Prefill 和 Decode 在同一个 GPU 上

问题:
  - Prefill 是 compute-bound (需要大量算力)
  - Decode 是 memory-bound (需要大量显存带宽)
  - 两种需求在一个 GPU 上冲突

PD 分离:
  P-Node: 专门做 prefill → 配备更多算力/高带宽 GPU
  D-Node: 专门做 decode → 配备大显存 GPU
  → 传输 protocol: 将 KV Cache 从 P 传到 D
```

### 2.2 SGLang PD 传输协议扩展

```
DeepSeek-V4 扩展了 SGLang 的 PD 传输协议:

原始 SGLang PD:
  传输完整的 KV Cache (dense attention)

V4 扩展:
  支持页索引 KV 传输 (page-indexed KV transfer)
  → 不是传输所有 KV, 而是只传输被引用的页
  → 与 ShadowRadix 的虚拟 token 槽 + 影子投影配合
  → P-Node 做完 prefill 后, 将影子映射传给 D-Node
  → D-Node 通过影子找到对应的物理 KV 页
```

## 3. CUDA Graph 内元数据准备

### 3.1 问题

```
混合注意力每个 decode step 需要:
  - SWA 页索引 (哪些物理页包含最近的 128 个 SWA token)
  - 影子映射 (每个虚拟槽在三个池中的物理位置)
  - 压缩器/索引器计划 (哪些 C4 位置需要被压缩和索引)
  - Per-pool 写入位置 (新 token 的 KV 写到哪)

传统做法 (Python):
  每次 forward 前, Python 遍历所有 running 请求
  构建上述元数据 → 传给 CUDA kernel
  → 每次 ~100us Python overhead → 小 batch 时成为瓶颈

CUDA Graph 的问题:
  CUDA Graph 录制时"冻结"了所有参数
  → 元数据在录制时固定, 不能改变
  → 但每个 step 的元数据是不同的!
```

### 3.2 解决方案: 图内元数据重建

```
DeepSeek-V4 的做法:

  录制 CUDA Graph 时, 不录制元数据 (作为 placeholder)
  而是在 Graph 重放时, 通过设备端的内核重建元数据:
  
  1. 一个专门的 "元数据准备" kernel
  2. 在设备端读取 block table / shadow map / pool state
  3. 计算当前 step 的元数据 (索引、计划等)
  4. 写入 Graph 的 placeholder 位置
  5. 后续的 attention kernel 直接使用
  
  → Python 端完全不需要参与 per-step 元数据准备!
  → 消除了 Python overhead
```

## 4. 分层多流重叠

### 4.1 两级流扇出

```
注意力准备阶段包含多个小 kernel:
  Q 投影 / KV 投影 / 压缩器 / 索引器

两级流扇出 (hierarchical multi-stream fan-out):

Level 1 (顶层): 4 个准备操作并行运行
  - Q 投影 + Q LoRA
  - KV 投影
  - 压缩器 (C4/C128)
  - 索引器 (C4 top-k)

Level 2 (索引器内部): 子拆分
  - 索引器的 histogram + threshold + scatter 进一步并行化

依赖传递: 使用 CUDA event 做细粒度同步
  - q_lora_ready: Q 投影完成后触发
  - q_scale_ready: Q scale 完成后触发
  → 依赖方等待特定 event 而非整个 stream

启用条件: 仅在 batch 较小时启用
  → 大 batch 时 kernel 本身已经足够大, overlap 收益小
  → 小 batch 时各 kernel 都很小, overlap 收益大
```

## 5. 生产部署配置

```
DeepSeek-V4 Pro:
  Hardware: 8 × B200 (TP=8)
  Engine: SGLang (with DeepSeek extensions)
  Context: up to 1M tokens
  Throughput: ~180 token/s @ 900K context (1 batch)
  
DeepSeek-V4 Flash:
  Hardware: 4 × H200 (TP=4) or 2 × B200
  Engine: SGLang
  Context: up to 1M tokens
  Throughput: ~240 token/s @ 900K context (1 batch)

关键部署参数:
  --enable-shadow-radix       # 开启 ShadowRadix
  --enable-hisparse            # 开启 HiSparse
  --speculative-model mtp-1    # 单层 MTP
  --max-context 1000000        # 1M context
  --pd-disaggregation          # PD 分离
  --cuda-graph-mode full       # 全 CUDA Graph
```

## 6. V4 系列总结

### 六大技术创新回顾

| # | 创新 | 解决的问题 | 影响范围 |
|---|------|----------|---------|
| 1 | Hybrid Sparse Attention | 1M 上下文下的 O(n^2) 计算瓶颈 | 架构 |
| 2 | mHC Connections | 深层网络的梯度流和表示质量 | 训练+推理 |
| 3 | FP4 Expert Weights | 带宽瓶颈 (decode 是 memory-bound) | 推理 |
| 4 | ShadowRadix | 混合注意力的前缀缓存 | 推理 |
| 5 | HiSparse | 长上下文的 GPU 显存瓶颈 | 推理 |
| 6 | 6D Parallelism | 1.6T 参数模型的训练 | 训练 |

### 未来方向 (推测)

```
1. FP4 扩展到全部权重 (不仅是专家)
2. 更激进的稀疏注意力 (更细粒度的 C4 top-k)
3. ShadowRadix 放到更多框架 (vLLM 等)
4. 更高效的投机解码 (EAGLE-like multi-layer)
5. 训练时即考虑推理约束 (train-inference co-design)
```